# **Diplomado IA: Audio y Video - Parte 1**. <br> Práctico 5: Aplicaciones 2
---
---

**Profesores:**
- Alain Raymond
- Gabriel Sepúlveda
- Álvaro Soto

**Ayudante:**
- Andreina Cota
---
---

# **Instrucciones Generales**

El siguiente práctico se debe realizar de forma individual. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Las secciones donde se planteen preguntas de forma explícita, deben ser respondida en celdas de texto, y no se aceptará solo el _output_ de una celda de código como respuesta.

**Nombre alumno:**
Vicente Zapata Concha

El siguiente práctico cuenta con secciones que contienen los experimentos presentados durante la sesión de laboratorio, y actividades que deberán ser desarrolladas y luego entregadas como tarea. En esta oportunidad, las actividades corresponden a preguntas de alternativa.

Antes de responder, se recomienda **fuertemente** revisar las secciones previas donde se desarrollan los ejemplos, dado que algunas de las actividades pueden ser completadas reutilizando el mismo código.

**Fecha de entrega:** domingo 24 de mayo de 2026, 23:59 hrs.

#Sources

**End-to-End Audiovisual Speech Recognition**

dataset: https://www.robots.ox.ac.uk/~vgg/data/lip_reading/lrw1.html

paper: https://arxiv.org/pdf/1802.06424.pdf

github: https://github.com/mpc001/end-to-end-lipreading

#Preámbulo

In [1]:
import sys
import os
import os.path
import glob
import math
import random
import numpy as np
import cv2
import librosa
import errno
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

In [2]:
!if [ ! -f Audiovisual.zip ]; then wget -q --show-progress https://www.dropbox.com/s/j6het8rq3j2ewng/Audiovisual.zip; fi
!if [ ! -f label_sorted.txt ]; then wget -q --show-progress https://www.dropbox.com/s/r44j8lhhsgjvjzb/label_sorted.txt; fi
!if [ ! -f lipread_testset_mini.tar.gz ]; then wget -q --show-progress https://www.dropbox.com/s/cbr5q72b8cef22i/lipread_testset_mini.tar.gz; fi
#!if [ ! -f lipread_testset.tar.gz ]; then wget -q --show-progress https://www.dropbox.com/s/4e3hkzaoizd491y/lipread_testset.tar.gz; fi
!unzip -q Audiovisual.zip
!tar xzf lipread_testset_mini.tar.gz
#!tar xzf lipread_testset.tar.gz

Audiovisual.zip     100%[===================>] 287.80M  63.2MB/s    in 5.0s    
label_sorted.txt    100%[===================>]   3.70K  --.-KB/s    in 0s      
lipread_testset_min 100%[===================>] 339.64M   117MB/s    in 2.9s    


#Preprocesamiento de datos

In [3]:
!cat label_sorted.txt | wc -l

500


In [4]:
!ls -l lipread_testset_mini/

total 2000
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABOUT
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABSOLUTELY
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABUSE
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCESS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCORDING
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCUSED
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACROSS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACTION
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACTUALLY
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFFAIRS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFFECTED
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFRICA
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFTER
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFTERNOON
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGAIN
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGAINST
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGREE
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGREEMENT
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AHEAD
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ALLEGATIONS
drwxrwxr-x

In [5]:
def extract_opencv(filename):
  video = []
  cap = cv2.VideoCapture(filename)
  while cap.isOpened():
    ret, frame = cap.read() # BGR
    if ret:
      video.append(frame)
    else:
      break
  cap.release()
  video = np.array(video)
  return video[...,::-1]

def video_converter(basedir, basedir_to_save):
  if not os.path.isdir( basedir_to_save ):
    os.makedirs( basedir_to_save, exist_ok = True )
  filenames = glob.glob(os.path.join(basedir, '*', '*', '*.mp4')) # <basedir>/<word>/<train, val, test>/<filename.mp4>
  for filename in filenames:
    data = extract_opencv(filename)[:, 115:211, 79:175]
    path_to_save = os.path.join(basedir_to_save,
                  filename.split('/')[-3],
                  filename.split('/')[-2],
                  filename.split('/')[-1][:-4]+'.npz')
    if not os.path.exists(os.path.dirname(path_to_save)):
      try:
        os.makedirs(os.path.dirname(path_to_save))
      except OSError as exc:
        if exc.errno != errno.EEXIST:
          raise
    np.savez(path_to_save, data=data)

def audio_converter(basedir, basedir_to_save):
  if not os.path.isdir( basedir_to_save ):
    os.makedirs( basedir_to_save, exist_ok = True )
  filenames = glob.glob(os.path.join(basedir, '*', '*', '*.mp4')) # <basedir>/<word>/<train, val, test>/<filename.mp4>
  for filename in filenames:
    data = librosa.load(filename, sr=16000)[0][-19456:]
    path_to_save = os.path.join(basedir_to_save,
                  filename.split('/')[-3],
                  filename.split('/')[-2],
                  filename.split('/')[-1][:-4]+'.npz')
    if not os.path.exists(os.path.dirname(path_to_save)):
      try:
        os.makedirs(os.path.dirname(path_to_save))
      except OSError as exc:
        if exc.errno != errno.EEXIST:
          raise
    np.savez( path_to_save, data=data)

In [6]:
video_converter('lipread_testset_mini', 'preprocessed/video')
audio_converter('lipread_testset_mini', 'preprocessed/audio')

Se han truncado las últimas 5000 líneas del flujo de salida.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1744/2457611317.py:37: UserWarning: PySoundFile failed. Trying audioread instead.
  data = librosa.load(filename, sr=16000)[0][-19456:]
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1744/2457611317.py:37: UserWarning: PySoundFile failed. Trying audioread instead.
  data = librosa.load(filename, sr=16000)[0][-19456:]
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duratio

#Dataloader

In [7]:
def load_audio_file(filename):
  return np.load(filename)['data']

def load_video_file(filename):
  cap = np.load(filename)['data']
  arrays = np.stack([cv2.cvtColor(cap[_], cv2.COLOR_RGB2GRAY) for _ in range(29)], axis=0)
  arrays = arrays / 255.
  return arrays

class MyDataset():
  def __init__(self, folds, audio_path, video_path):
    '''
      folds: partition type -> test, train, val
      audio_path: ruta a archivos de audio numpy
      video_path: ruta a archivos de video numpy
    '''
    self.folds = folds
    self.audio_path = audio_path
    self.video_path = video_path
    self.clean = 1 / 7.
    with open('label_sorted.txt') as myfile:
      self.data_dir = myfile.read().splitlines()
    self.filenames = glob.glob(os.path.join(self.audio_path, '*', self.folds, '*.npz'))
    self.list = {}
    for i, x in enumerate(self.filenames):
      target = x.split('/')[-3]
      for j, elem in enumerate(self.data_dir):
        if elem == target:
          self.list[i] = [x]
          self.list[i].append(j)

  def normalisation(self, inputs):
    inputs_std = np.std(inputs)
    if inputs_std == 0.:
      inputs_std = 1.
    return (inputs - np.mean(inputs))/inputs_std

  def __getitem__(self, idx):
    video_inputs = load_video_file(os.path.join(self.video_path,
                          self.list[idx][0].split('/')[-3],
                          self.list[idx][0].split('/')[-2],
                          self.list[idx][0].split('/')[-1][:-4]+'.npz'))
    self.list[idx][0] = self.list[idx][0]
    audio_inputs = load_audio_file(self.list[idx][0])
    audio_inputs = self.normalisation(audio_inputs)
    labels = self.list[idx][1]
    return audio_inputs, video_inputs, labels

  def __len__(self):
    return len(self.filenames)

In [8]:
def data_loader(datatype, audio_dataset, video_dataset, batch_size):
  dsets = MyDataset(datatype, audio_dataset, video_dataset)
  dset_loader = torch.utils.data.DataLoader(dsets, batch_size = batch_size, shuffle=True, num_workers=4)
  dset_size = len(dsets)
  print('\nStatistics: {}: {}'.format(datatype, dset_size))
  return dset_loader, dset_size

#Modelo

##Bloques genéricos

In [9]:
class GRU(nn.Module):

  def __init__(self, input_size, hidden_size, num_layers, num_classes, output_layer=False, every_frame=False):
    super(GRU, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.output_layer = output_layer
    self.every_frame = every_frame
    self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
    self.fc = nn.Linear(hidden_size*2, num_classes)

  def to( self, device ):
    self.device = device
    return super( GRU, self ).to( device )

  def forward(self, x):
    h0 = Variable(torch.zeros(self.num_layers*2, x.size(0), self.hidden_size).to(self.device))
    # Forward propagate RNN
    out, _ = self.gru(x, h0)
    if self.output_layer:
      if self.every_frame:
        out = self.fc(out)  # predictions based on every time step
      else:
        out = self.fc(out[:, -1, :])  # predictions based on last time-step
    return out

##Modelo de audio


In [10]:
class BasicBlock1D(nn.Module):

  def __init__(self, inplanes, planes, stride=1, downsample=None):
    super(BasicBlock1D, self).__init__()
    self.downsample = downsample
    self.stride = stride

    self.conv1 = nn.Conv1d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm1d(planes)
    self.relu = nn.ReLU(inplace=True)

    self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm1d(planes)

  def forward(self, x):
    residual = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
      residual = self.downsample(x)
    out += residual
    out = self.relu(out)
    return out


class ResNet(nn.Module):

  def __init__(self, block, layers, num_classes=1000):
    self.inplanes = 64
    super(ResNet, self).__init__()
    self.layer1 = self._make_layer(block, 64, layers[0])
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
    self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
    self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
    self.avgpool = nn.AvgPool1d(kernel_size=21, padding=1)
    self.fc = nn.Linear(512, num_classes)

  def _make_layer(self, block, planes, blocks, stride=1):
    downsample = None
    if stride != 1 or self.inplanes != planes:
      downsample = nn.Sequential(
        nn.Conv1d(self.inplanes, planes,
              kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm1d(planes),
      )

    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes
    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = x.transpose(1, 2)
    x = x.contiguous()
    x = x.view(-1, x.size(2))
    x = self.fc(x)
    return x


class AudioLipreading(nn.Module):
  def __init__(self, inputDim=256, hiddenDim=512, nClasses=500, frameLen=29):
    super(AudioLipreading, self).__init__()
    self.inputDim = inputDim
    self.hiddenDim = hiddenDim
    self.nClasses = nClasses
    self.frameLen = frameLen
    self.nLayers = 2
    # frontend1D
    self.fronted1D = nn.Sequential(
        nn.Conv1d(1, 64, kernel_size=80, stride=4, padding=38, bias=False),
        nn.BatchNorm1d(64),
        nn.ReLU(True)
        )
    # resnet
    self.resnet18 = ResNet(BasicBlock1D, [2, 2, 2, 2], num_classes=self.inputDim)
    # backend_gru
    self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses)

  def to( self, device ):
    self.gru.to(device)
    return super( AudioLipreading, self ).to( device )

  def forward(self, x):
    x = x.view(-1, 1, x.size(1))
    x = self.fronted1D(x)
    x = x.contiguous()
    x = self.resnet18(x)
    x = x.view(-1, self.frameLen, self.inputDim)
    x = self.gru(x)
    return x

##Modelo de video

In [11]:
class BasicBlock2D(nn.Module):

  def __init__(self, inplanes, planes, stride=1, downsample=None):
    super(BasicBlock2D, self).__init__()
    self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(planes)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(planes)
    self.downsample = downsample
    self.stride = stride

  def forward(self, x):
    residual = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
      residual = self.downsample(x)
    out += residual
    out = self.relu(out)
    return out


class ResNet2D(nn.Module):

  def __init__(self, block, layers, num_classes=1000):
    self.inplanes = 64
    super(ResNet2D, self).__init__()
    self.layer1 = self._make_layer(block, 64, layers[0])
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
    self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
    self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
    self.avgpool = nn.AvgPool2d(2)
    self.fc = nn.Linear(512, num_classes)
    self.bnfc = nn.BatchNorm1d(num_classes)

  def _make_layer(self, block, planes, blocks, stride=1):
    downsample = None
    if stride != 1 or self.inplanes != planes:
      downsample = nn.Sequential(
        nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm2d(planes),
      )

    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes
    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)
    x = self.bnfc(x)
    return x


class VideoLipreading(nn.Module):

  def __init__(self, inputDim=256, hiddenDim=512, nClasses=500, frameLen=29):
    super(VideoLipreading, self).__init__()
    self.inputDim = inputDim
    self.hiddenDim = hiddenDim
    self.nClasses = nClasses
    self.frameLen = frameLen
    self.nLayers = 2
    # frontend3D
    self.frontend3D = nn.Sequential(
        nn.Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
        nn.BatchNorm3d(64),
        nn.ReLU(True),
        nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )
    # resnet
    self.resnet34 = ResNet2D(BasicBlock2D, [3, 4, 6, 3], num_classes=self.inputDim)
    # backend_gru
    self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses)

  def to( self, device ):
    self.gru.to(device)
    return super( VideoLipreading, self ).to( device )

  def forward(self, x):
    x = self.frontend3D(x)
    x = x.transpose(1, 2)
    x = x.contiguous()
    x = x.view(-1, 64, x.size(3), x.size(4))
    x = self.resnet34(x)
    x = x.view(-1, self.frameLen, self.inputDim)
    x = self.gru(x)
    return x

#Evaluación

In [12]:
def CenterCrop(batch_img, size):
  w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
  th, tw = size
  img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
  for i in range(len(batch_img)):
    x1 = int(round((w - tw))/2.)
    y1 = int(round((h - th))/2.)
    img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
  return img

def ColorNormalize(batch_img):
  mean = 0.413621
  std = 0.1700239
  batch_img = (batch_img - mean) / std
  return batch_img

In [13]:
def reload_model(model, path=""):
  model_dict = model.state_dict()
  pretrained_dict = torch.load(path)
  pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}
  model_dict.update(pretrained_dict)
  model.load_state_dict(model_dict)
  print('*** model has been successfully loaded! ***')
  return model

In [14]:
device = torch.device( 'cuda' if torch.cuda.is_available() else 'cpu' )
print( 'running on: %s' % (device) )

every_frame = True
audio_model = AudioLipreading(inputDim=512, hiddenDim=512, nClasses=500, frameLen=29)
video_model = VideoLipreading(inputDim=256, hiddenDim=512, nClasses=500, frameLen=29)
concat_model = GRU(2048, 512, 2, 500, output_layer=True, every_frame=every_frame)

# reload model
print('reload audio model')
audio_model = reload_model(audio_model, 'Audiovisual/Audiovisual_a_part.pt')
print("reload video model")
video_model = reload_model(video_model, 'Audiovisual/Audiovisual_v_part.pt')
print("reload LSTM model")
concat_model = reload_model(concat_model, 'Audiovisual/Audiovisual_c_part.pt')

audio_model = audio_model.to( device )
video_model = video_model.to( device )
concat_model = concat_model.to( device )

running on: cuda
reload audio model
*** model has been successfully loaded! ***
reload video model
*** model has been successfully loaded! ***
reload LSTM model
*** model has been successfully loaded! ***


In [15]:
dset_loader, dset_size = data_loader('test', 'preprocessed/audio', 'preprocessed/video', batch_size = 1)


Statistics: test: 2500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [16]:
audio_model.eval()
video_model.eval()
concat_model.eval()

running_loss = 0.0
running_corrects = 0.0
running_all = 0.0
with torch.no_grad():
  for batch_idx, (audio_inputs, video_inputs, targets) in enumerate(dset_loader):
    batch_img = CenterCrop(video_inputs.numpy(), (88, 88))
    batch_img = ColorNormalize(batch_img)

    batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
    video_inputs = torch.from_numpy(batch_img)
    video_inputs = video_inputs.float().permute(0, 4, 1, 2, 3)

    audio_inputs = audio_inputs.float()

    audio_inputs = audio_inputs.to( device )
    video_inputs = video_inputs.to( device )
    targets = targets.to( device )

    audio_outputs = audio_model(audio_inputs)
    video_outputs = video_model(video_inputs)
    inputs = torch.cat((audio_outputs, video_outputs), dim=2)
    outputs = concat_model(inputs)

    if every_frame:
      outputs = torch.mean(outputs, 1) # average probability among frames
    _, preds = torch.max(F.softmax(outputs, dim=1).data, 1)

    #running_loss += loss.data[0] * inputs.size(0)
    running_corrects += torch.sum(preds == targets.data)
    running_all += len(inputs)
print('Accuracy: {:.4f}'.format(running_corrects / len(dset_loader.dataset))+'\n')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Accuracy: 0.9884



##Evaluación cualitativa

In [17]:
def normalisation(inputs):
  inputs_std = np.std(inputs)
  if inputs_std == 0.:
    inputs_std = 1.
  return (inputs - np.mean(inputs))/inputs_std

def predict(filename):
  # data loading
  npy_basename = 'preprocessed'
  with open('label_sorted.txt') as myfile:
    id2label = myfile.read().splitlines()
    label2id = { klass:i for i, klass in enumerate(id2label) }
  audio_input = load_audio_file( os.path.join( npy_basename,
                                               'audio',
                                                filename.split('/')[-3],
                                                filename.split('/')[-2],
                                                filename.split('/')[-1][:-4]+'.npz' ) )
  audio_input = normalisation(audio_input)
  video_input = load_video_file( os.path.join( npy_basename,
                                               'video',
                                                filename.split('/')[-3],
                                                filename.split('/')[-2],
                                                filename.split('/')[-1][:-4]+'.npz' ) )
  label = filename.split('/')[-3]
  label_id = label2id[label]

  audio_input = np.expand_dims( audio_input, 0 ) # add batch dimension
  video_input = np.expand_dims( video_input, 0 ) # add batch dimension

  # prediction
  batch_img = CenterCrop(video_input, (88, 88))
  batch_img = ColorNormalize(batch_img)
  batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
  video_input = torch.from_numpy(batch_img)
  video_input = video_input.float().permute(0, 4, 1, 2, 3)


  audio_input = torch.from_numpy(audio_input).float()

  audio_input = audio_input.to( device )
  video_input = video_input.to( device )

  audio_output = audio_model(audio_input)
  video_output = video_model(video_input)
  input = torch.cat((audio_output, video_output), dim=2)
  output = concat_model(input)

  if every_frame:
    output = torch.mean(output, 1) # average probability among frames
  _, pred = torch.max(F.softmax(output, dim=1).data, 1)

  pred_str = id2label[int(pred)]

  return preds, pred_str

In [18]:
video_filename = 'lipread_testset_mini/AMERICAN/test/AMERICAN_00001.mp4'
#video_filename = 'lipread_testset_mini/CHILDREN/test/CHILDREN_00001.mp4'
#video_filename = 'lipread_testset_mini/EXAMPLE/test/EXAMPLE_00001.mp4'
#video_filename = 'lipread_testset_mini/MAKING/test/MAKING_00001.mp4'
#video_filename = 'lipread_testset_mini/YESTERDAY/test/YESTERDAY_00001.mp4'

pred, pred_str = predict(video_filename)
print( 'Model prediction: %s [%d]' % (pred_str, int(pred)) )

Model prediction: AMERICAN [307]


In [19]:
from IPython.display import HTML
from base64 import b64encode
# Convert mp4 to format supported by Colab
os.system(f"ffmpeg -i {video_filename} -vcodec libx264 {os.path.basename(video_filename)}")
mp4 = open(os.path.basename(video_filename),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<h1>Predicted word: %s</h1>
<br>
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % (pred_str, data_url))

#Actividades

---
## Contexto — Actividad 1: ¿Por qué usar video en Speech Recognition?

El laboratorio implementa un sistema de **reconocimiento de voz audiovisual** (Audiovisual Speech Recognition, AVSR).
A diferencia del ASR clásico que solo usa audio, aquí el modelo también procesa los frames del video.

Esto se conoce coloquialmente como **"leer los labios con IA"** (*lip reading*).

### ¿Por qué combinar audio + video?

Imaginemos dos situaciones:

| Situación | Solo audio | Audio + Video (AVSR) |
|---|---|---|
| Ambiente silencioso | ✅ Funciona bien | ✅ Funciona bien |
| Ruido de fondo intenso (calle, fiesta, viento) | ❌ El modelo falla mucho | ✅ Los labios siguen visibles y compensan |
| Audio muy bajo o distorsionado | ❌ Señal ininteligible | ✅ El modelo se apoya en el video |

Este fenómeno fue estudiado originalmente por el psicólogo **McGurk (1976)**: cuando el audio y el video no coinciden,
el cerebro humano percibe una tercera sílaba. Nuestro cerebro **ya integra ambas modalidades de forma natural**.

El AVSR replica este comportamiento: ante ruido, los labios aportan la información que el audio pierde.

> **Pregunta**: ¿Cuándo se vuelve *necesario* usar los frames de video?

##Actividad 1

¿ Por qué puede ser necesario utilizar los frames de video para la tarea de speech recognition ?

In [20]:
Respuesta = 'Para dar robustez al modelo cuando el audio viene con ruido ambiente' #@param ["seleccione una opcion", "Los frames de video son utilizados para leer los labios y generar el audio del habla","Para dar robustez al modelo cuando el audio viene con ruido ambiente", "Para localizar la persona que está hablando dentro de la imagen", "Para determinar el intervalo de tiempo donde se produce el habla", "Los frames de video son imprescindibles para reconocer la palabra pronunciada"]

### Explicación — Actividad 1

**Respuesta correcta**: `Para dar robustez al modelo cuando el audio viene con ruido ambiente`

La palabra clave en la pregunta es **"necesario"**. El video no siempre es indispensable —
en un ambiente silencioso, el audio solo es suficiente.

Pero cuando hay **ruido de fondo** (señal-ruido baja), el canal de audio se degrada y el modelo
necesita una fuente de información alternativa. Ahí es donde los frames del video (los movimientos
de los labios) aportan la información faltante.

¿Por qué las otras opciones son incorrectas?

| Opción | ¿Por qué es incorrecta? |
|---|---|
| *Leer labios para generar el audio* | Eso sería síntesis de voz (TTS), no reconocimiento |
| *Localizar a la persona que habla* | Útil pero no es el objetivo principal del módulo de video |
| *Determinar el intervalo de tiempo del habla* | Eso es detección de actividad vocal (VAD), diferente tarea |
| *Video es imprescindible para reconocer palabras* | Demasiado fuerte: el ASR sin video funciona bien en ambientes limpios |

> 💡 **Para recordar**: El AVSR fue diseñado especialmente para escenarios ruidosos.
> En silencio, el audio solo ya es suficiente.


---
## Contexto — Actividad 2: Entrenamiento por etapas

El modelo del laboratorio tiene una arquitectura compleja con varias sub-redes:

```
Audio  ──► ResNet 1D ──► BiGRU ──┐
                                  ├──► Fusión ──► Softmax ──► Palabra
Video  ──► ResNet 2D ──► BiGRU ──┘
```

El entrenamiento se hace en **3 etapas sucesivas**, no todo junto desde el principio:

### Etapas de entrenamiento

| Etapa | Qué se entrena | Objetivo |
|---|---|---|
| **Etapa 1** | ResNets (audio + video por separado) | Aprender representaciones de bajo nivel por modalidad |
| **Etapa 2** | BiGRUs (encima de las ResNets congeladas) | Aprender patrones temporales de secuencias |
| **Etapa 3** | Todo el modelo junto (fine-tuning) | Afinar la fusión conjunta de ambas modalidades |

Esta estrategia se llama **curriculum learning** o **staged training**.

> **Pregunta**: ¿Cuál es la razón principal para hacer este entrenamiento escalonado?

##Actividad 2

¿ Por qué el entrenamiento del modelo se hace por etapas ? ( primero ResNets, luego BiGRUs, luego todo junto ).

In [21]:
Respuesta = 'Para aumentar la estabilidad del entrenamiento y obtener un mayor rendimiento' #@param ["seleccione una opcion", "Porque el modelo es muy grande y no es posible alamcenar el gradiente de todos sus pesos en una GPU","Porque no es posible combinar modelos feedforward con modelos recurrentes en la propagación de gradientes", "Para separar el entrenamiento del stream de video del de audio", "Para aumentar la estabilidad del entrenamiento y obtener un mayor rendimiento", "Para separar cada uno de los 29 instantes de tiempo que componen los videos de entrada"]

### Explicación — Actividad 2

**Respuesta correcta**: `Para aumentar la estabilidad del entrenamiento y obtener un mayor rendimiento`

Entrenar un modelo multimodal grande de principio a fin (*end-to-end*) de forma directa es **extremadamente inestable**.
Los gradientes provenientes de ramas distintas (audio vs. video) pueden tener escalas muy diferentes,
lo que genera actualizaciones de pesos caóticas.

Al entrenar por etapas:
1. Primero cada ResNet aprende a extraer buenos features **por su cuenta** (sin la presión de optimizar la fusión).
2. Luego las BiGRUs aprenden a modelar la secuencia temporal con features ya estables.
3. Finalmente, el fine-tuning conjunto ajusta finamente la colaboración entre ramas.

El resultado es un entrenamiento **más estable** y un modelo **más preciso** al final.

¿Por qué las otras opciones son incorrectas?

| Opción | ¿Por qué es incorrecta? |
|---|---|
| *Modelo muy grande, no cabe en GPU* | Con gradient checkpointing se puede manejar; no es la razón principal |
| *No es posible combinar feedforward + recurrente* | Sí es posible, PyTorch lo hace sin problema |
| *Separar entrenamiento de video y audio* | Las etapas no son por modalidad sino por tipo de bloque (CNN vs. RNN) |
| *Separar los 29 instantes de tiempo* | Los 29 frames se procesan como secuencia, no por separado |

> 💡 **Para recordar**: El staged training es una técnica clave en modelos multimodales grandes.
> Permite que cada parte del modelo converja antes de intentar optimizar el sistema completo.


---
## Contexto — Actividad 3: Capa Convolucional 3D en la rama de video

En la rama que procesa el video, el primer bloque no es una convolución 2D (espacial) sino una **convolución 3D**.

### ¿Qué diferencia hay entre Conv2D y Conv3D?

| Tipo | Dimensiones del kernel | Opera sobre |
|---|---|---|
| **Conv2D** | (H × W) | Una imagen estática — detecta bordes, texturas, formas |
| **Conv3D** | (T × H × W) | Un volumen espacio-temporal — detecta **movimiento** |

### Entrada a la rama de video

El video de entrada tiene forma:

$$\text{input} \in \mathbb{R}^{\text{batch} \times 1 \times T \times H \times W}$$

Donde $T = 29$ frames, $H \times W$ es la resolución de cada frame (recortada en la región bucal).

La convolución 3D tiene un kernel de tamaño $(5 \times 7 \times 7)$ —
es decir, analiza **5 frames consecutivos** a la vez en una ventana espacial de 7×7 píxeles.

> **Pregunta**: ¿Cuál es el objetivo principal de esta capa 3D?
> *(Hint: sección 3.1 del paper de Afouras et al.)*

##Actividad 3

Como vimos en clases, para la *rama* que se encarga de procesar el video, se comienza por agregar una capa convolucional 3D. ¿ Cuál es el principal objetivo de esta capa ?

**Hint:** Si no se acuerda, puede descargar el paper y leer la sección 3.1, página 2.

In [22]:
Respuesta = 'Capturar las dinámicas producidas en pequeños intervalos de tiempo' #@param ["seleccione una opcion", "Reducir la dimensión temporal desde 29 frames a 1 que resuma todo el movimiento","Reducir los 3 canales RGB de entrada a una matriz bidimensional", "Realizar un downsampling de algunos frames que permitan recuperar patrones temporales", "Capturar las dinámicas producidas en pequeños intervalos de tiempo"]

### Explicación — Actividad 3

**Respuesta correcta**: `Capturar las dinámicas producidas en pequeños intervalos de tiempo`

La capa convolucional 3D con kernel $(5 \times 7 \times 7)$ tiene como objetivo principal
**capturar el movimiento local de los labios** en pequeñas ventanas temporales de 5 frames.

Esto es fundamental porque:
- Las palabras se articulan a través de **cambios rápidos** en la forma de los labios.
- Una sola imagen (frame) no revela la palabra — es la **transición entre frames** lo que la define.
- Con solo Conv2D (por frame), el modelo no vería el movimiento. Con Conv3D, el modelo aprende
  que, por ejemplo, "BA" implica los labios cerrándose y abriéndose en pocos frames.

```
Frames:   t-2  t-1   t   t+1  t+2
          [😮][😮][😐][🫦][😶]  ← kernel 3D (tamaño T=5) captura esta secuencia
                ↓
         Feature map espacio-temporal
         (aprende el movimiento del labio en esa ventana)
```

¿Por qué las otras opciones son incorrectas?

| Opción | ¿Por qué es incorrecta? |
|---|---|
| *Reducir de 29 frames a 1* | El kernel es (5×7×7) con stride=1×2×2: reduce dimensión espacial, no colapsa todos los frames en 1 |
| *Reducir 3 canales RGB a 2D* | El video de entrada ya viene en escala de grises (1 canal); no es la motivación |
| *Downsampling de frames para patrones temporales* | La reducción temporal es moderada (no elimina la mayoría de frames); el objetivo es la captura de movimiento |

> 💡 **Para recordar**: Conv3D = Conv2D + dimensión temporal. Es el componente clave que
> le da al modelo la capacidad de "ver" el movimiento en lugar de solo imágenes estáticas.
